In [0]:
 %sql
CREATE CONNECTION IF NOT EXISTS project_earthquake_conn
TYPE HTTP
OPTIONS
(
  host = "https://earthquake.usgs.gov",
  port = 443,
  base_path = "/earthquakes/feed/v1.0/",
  bearer_token = "NA"
)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("project_earthquake_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"
print(base_url)
# print(conn)

In [0]:
import requests
import json
url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)
response.json()

In [0]:
dbutils.widgets.text('catalog_name', 'project_development', 'project_development')
catalog_name = dbutils.widgets.get('catalog_name')
catalog_name

In [0]:


#THE TASK IS TO SAVE THE ABOVE JSON DATA INTO OUR BRNZE LAYER
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA bronze")
# USE catalog project_development;
# USE schema bronze;
spark.sql("CREATE VOLUME IF NOT EXISTS earthquake_data_1")


In [0]:
import datetime 
current_date = datetime.datetime.now().strftime("%d-%m-%Y")
current_date

In [0]:
bronze_vol_path = f"/Volumes/{catalog_name}/bronze/earthquake_data_1/{current_date}_earthquake_data.json"

In [0]:
data = response.json() #from 3rd cell
dbutils.fs.put(bronze_vol_path, json.dumps(data), overwrite=True)